In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# 1. Load the engineered dataset
print("Step 1: Loading 'Model1_Demand_Data.csv'...")
df = pd.read_csv('Model1_Demand_Data.csv')
print(f"Dataset loaded. Total shifts for training: {len(df)}")

Step 1: Loading 'Model1_Demand_Data.csv'...
Dataset loaded. Total shifts for training: 1623


In [4]:
df.columns

Index(['transaction_date', 'store_location', 'month', 'day_of_week',
       'is_weekend', 'shift', 'shift_num', 'total_cups', 'prev_cups'],
      dtype='str')

In [6]:
avg_cups = df['total_cups'].mean()
print(f"Average cups: {avg_cups}")

Average cups: 132.1441774491682


In [8]:
# Calculate the mean across 4 different dimensions
multi_avg = df.groupby(['day_of_week', 'is_weekend', 'store_location', 'shift'])['total_cups'].mean()

# Use .reset_index() to convert it back into a clean, flat table (DataFrame)
multi_avg_df = multi_avg.reset_index()

print(multi_avg_df)

   day_of_week  is_weekend   store_location      shift  total_cups
0       Friday           0          Astoria  Afternoon  131.923077
1       Friday           0          Astoria    Evening   83.038462
2       Friday           0          Astoria    Morning  182.807692
3       Friday           0   Hell's Kitchen  Afternoon  101.230769
4       Friday           0   Hell's Kitchen    Evening   68.923077
..         ...         ...              ...        ...         ...
58   Wednesday           0   Hell's Kitchen    Evening   66.461538
59   Wednesday           0   Hell's Kitchen    Morning  218.807692
60   Wednesday           0  Lower Manhattan  Afternoon  113.538462
61   Wednesday           0  Lower Manhattan    Evening   34.600000
62   Wednesday           0  Lower Manhattan    Morning  241.192308

[63 rows x 5 columns]


In [11]:
# 1. Filter for Saturday only
saturday_data = df[df['day_of_week'] == 'Saturday']

# 2. Group by the specific dimensions to see shift performance
saturday_averages = saturday_data.groupby(['store_location', 'shift', 'is_weekend'])['total_cups'].mean().reset_index()

# 3. Sort by total_cups to see which Saturday shift is the "Final Boss" of demand
print(saturday_averages.sort_values(by='total_cups', ascending=False))

    store_location      shift  is_weekend  total_cups
8  Lower Manhattan    Morning           1  250.040000
5   Hell's Kitchen    Morning           1  220.280000
2          Astoria    Morning           1  171.200000
0          Astoria  Afternoon           1  136.320000
6  Lower Manhattan  Afternoon           1  121.480000
3   Hell's Kitchen  Afternoon           1  104.000000
1          Astoria    Evening           1   81.280000
4   Hell's Kitchen    Evening           1   66.640000
7  Lower Manhattan    Evening           1   34.708333


In [12]:
df = pd.read_csv('Model2_Revenue_Data.csv')
print(f"Dataset loaded. Total shifts for training: {len(df)}")

Dataset loaded. Total shifts for training: 1623


In [14]:
df.columns

Index(['transaction_date', 'store_location', 'month', 'day_of_week',
       'is_weekend', 'shift', 'shift_num', 'total_cups', 'total_revenue',
       'prev_revenue', 'prev_cups'],
      dtype='str')

In [16]:
# 1. Filter for Saturday only
saturday_data = df[df['day_of_week'] == 'Saturday']

# 2. Select both columns to average
# We use a list [['total_cups', 'total_revenue']] to tell Pandas to calculate both
saturday_stats = saturday_data.groupby(['store_location', 'shift', 'is_weekend'])[['total_cups', 'total_revenue']].mean().reset_index()

# 3. View the results
print(saturday_stats)

    store_location      shift  is_weekend  total_cups  total_revenue
0          Astoria  Afternoon           1  136.320000     440.142000
1          Astoria    Evening           1   81.280000     268.379200
2          Astoria    Morning           1  171.200000     565.316000
3   Hell's Kitchen  Afternoon           1  104.000000     327.826400
4   Hell's Kitchen    Evening           1   66.640000     208.506400
5   Hell's Kitchen    Morning           1  220.280000     767.636400
6  Lower Manhattan  Afternoon           1  121.480000     388.284400
7  Lower Manhattan    Evening           1   34.708333     107.272917
8  Lower Manhattan    Morning           1  250.040000     806.706400


In [17]:
df['store_location'].unique()

<StringArray>
['Astoria', 'Hell's Kitchen', 'Lower Manhattan']
Length: 3, dtype: str